# Objective

To develop a content-based Movie Recommendation System that suggests five movies similar to a user's selected movie by analyzing movie genres. The project uses One-Hot Encoding to convert categorical genre information into numerical features and the K-Nearest Neighbors (KNN) algorithm with Cosine Distance to identify movies with similar genres. The trained recommendation model is integrated with a Streamlit web application to provide an interactive movie discovery experience.

# Dataset Description

The dataset contains movie-related information used to build the content-based recommendation system.

In this project, the following features are selected for recommendation:

- **id:** Unique identifier of the movie.
- **title:** Name of the movie.
- **genre:** Genre or category of the movie, such as Action, Comedy, Drama, Crime, and Romance.

The genre column contains multiple categories for each movie. These genres are converted into numerical features using One-Hot Encoding so that the KNN algorithm can compare movies based on their genre information.

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Loading the Dataset

The movie dataset is loaded into a Pandas DataFrame using the `pd.read_csv()` function.

The `movies.head()` function displays the first five rows of the dataset. The required columns, including id, title, and genre, are selected for building the recommendation system.

This initial inspection helps us understand the structure of the dataset and the information available for movie recommendations.

In [21]:
movies = pd.read_csv('dataset.csv')

movies.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


# Data Exploration

Data exploration is performed to understand the dataset before preprocessing.

The `shape` function is used to identify the total number of rows and columns. The `info()` function provides information about column names, data types, and non-null values. The `describe()` function displays statistical information about the dataset.

Missing values and duplicate records are also checked to identify data quality issues.

In [22]:
movies = movies[['id', 'title', 'genre']]

movies.head(10)

,id,title,genre
0,278,The Shawshank Redemption,"Drama,Crime"
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance"
2,238,The Godfather,"Drama,Crime"
3,424,Schindler's List,"Drama,History,War"
4,240,The Godfather: Part II,"Drama,Crime"
5,667257,Impossible Things,"Family,Drama"
6,129,Spirited Away,"Animation,Family,Fantasy"
7,730154,Your Eyes Tell,"Romance,Drama"
8,372754,Dou kyu sei – Classmates,"Romance,Animation"
9,372058,Your Name.,"Romance,Animation,Drama"


In [23]:
print("Dataset Shape:", movies.shape)

print("\nDataset Information:")
movies.info()

print("\nStatistical Description:")
movies.describe(include='all')

Dataset Shape: (10000, 3)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      10000 non-null  int64
 1   title   10000 non-null  str  
 2   genre   9997 non-null   str  
dtypes: int64(1), str(2)
memory usage: 574.4 KB

Statistical Description:


,id,title,genre
count,10000.000000,10000,9997
unique,NaN,9661,2123
top,NaN,Beauty and the Beast,Comedy
freq,NaN,4,744
mean,161243.505000,NaN,NaN
std,211422.046043,NaN,NaN
min,5.000000,NaN,NaN
25%,10127.750000,NaN,NaN
50%,30002.500000,NaN,NaN
75%,310133.500000,NaN,NaN


# Data Preprocessing and Feature Selection

Data preprocessing is performed to clean the dataset and prepare it for model development.

The dataset contains 3 missing values in the genre column. These rows are removed using the `dropna()` function. Duplicate movie IDs are removed using `drop_duplicates()` to maintain unique movie records.

After cleaning, the dataset index is reset using `reset_index()`.

For feature selection, the movie title and genre information are retained. The genre column is selected as the main feature for generating movie recommendations.

In [24]:
print("Missing Values:")
print(movies.isnull().sum())

print("\nDuplicate Records:")
print(movies.duplicated().sum())

Missing Values:
id       0
title    0
genre    3
dtype: int64

Duplicate Records:
0


In [25]:
# Check missing values
print("Before preprocessing:")
print(movies.isnull().sum())

# Remove rows with missing title or genre
movies = movies.dropna(subset=['title', 'genre'])

# Remove duplicate movie IDs
movies = movies.drop_duplicates(subset=['id'])

# Reset index
movies = movies.reset_index(drop=True)

print("\nAfter preprocessing:")
print("Dataset Shape:", movies.shape)
print(movies.isnull().sum())

Before preprocessing:
id       0
title    0
genre    3
dtype: int64

After preprocessing:
Dataset Shape: (9997, 3)
id       0
title    0
genre    0
dtype: int64


# Genre Encoding Using One-Hot Encoding

Machine Learning algorithms require numerical data for performing calculations. Since movie genres are stored in text format, they must be converted into numerical representations.

In this project, `MultiLabelBinarizer` is used to perform One-Hot Encoding on movie genres.

Each unique genre is converted into a separate binary feature column. A value of 1 indicates that the movie belongs to that genre, whereas 0 indicates that the movie does not belong to that genre.

The resulting numerical feature matrix is used by the KNN algorithm to identify movies with similar genres.

In [26]:
from sklearn.preprocessing import MultiLabelBinarizer

# Split genres into lists
movies['genre_list'] = movies['genre'].apply(
    lambda x: [g.strip() for g in x.split(',')]
)

# Apply One-Hot Encoding
mlb = MultiLabelBinarizer()

genre_features = mlb.fit_transform(movies['genre_list'])

# Create feature DataFrame
genre_df = pd.DataFrame(
    genre_features,
    columns=mlb.classes_
)

print("Genre Features:")
print(genre_df.head(10))

print("\nTotal Genres:", len(mlb.classes_))
print("Feature Matrix Shape:", genre_features.shape)

Genre Features:
   Action  Adventure  Animation  Comedy  Crime  Drama  Family  Fantasy  \
0       0          0          0       0      1      1       0        0   
1       0          0          0       1      0      1       0        0   
2       0          0          0       0      1      1       0        0   
3       0          0          0       0      0      1       0        0   
4       0          0          0       0      1      1       0        0   
5       0          0          0       0      0      1       1        0   
6       0          0          1       0      0      0       1        1   
7       0          0          0       0      0      1       0        0   
8       0          0          1       0      0      0       0        0   
9       0          0          1       0      0      1       0        0   

   History  Horror  Music  Mystery  Romance  Science Fiction  TV Movie  \
0        0       0      0        0        0                0         0   
1        0       0   

# Feature Selection (X and Y)

Feature selection identifies the input data used by the recommendation model.

In this project, X represents the numerical genre feature matrix generated through One-Hot Encoding.

The Y target variable is not applicable because this project uses an unsupervised nearest-neighbor approach. The model does not predict movie ratings or classify movies into predefined categories.

The KNN algorithm uses the input genre features to find similar movies.

In [27]:
# X: Input Features
X = genre_features

print("X Shape:", X.shape)
print("Total Input Features:", X.shape[1])

# Y: Target Variable
Y = None

print("Y (Target Variable): Not Applicable")

X Shape: (9997, 18)
Total Input Features: 18
Y (Target Variable): Not Applicable


In [28]:
print("Original Dataset:")
display(movies[['title', 'genre']].head())

print("\nEncoded Dataset:")
display(genre_df.head())

Original Dataset:


,title,genre
0,The Shawshank Redemption,"Drama,Crime"
1,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance"
2,The Godfather,"Drama,Crime"
3,Schindler's List,"Drama,History,War"
4,The Godfather: Part II,"Drama,Crime"



Encoded Dataset:


,Action,Adventure,Animation,Comedy,Crime,Drama,Family,Fantasy,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0
4,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0


# KNN Model Building and Training

The K-Nearest Neighbors (KNN) algorithm is used to find movies that have similar genre features.

The `NearestNeighbors` class from Scikit-learn is used to create the KNN model.

The model is configured with 6 neighbors, Cosine Distance as the distance metric, and the Brute Force algorithm.

The `fit()` function is used to fit the model on the encoded genre feature matrix. After fitting, the model can identify the nearest movies based on genre similarity.

The selected movie itself is included in the six nearest results, so it is skipped to display the top five recommendations.

In [29]:
from sklearn.neighbors import NearestNeighbors

# Create KNN model
knn_model = NearestNeighbors(
    n_neighbors=6,
    metric='cosine',
    algorithm='brute'
)

# Train model using genre features
knn_model.fit(genre_features)

print("KNN Model trained successfully!")

KNN Model trained successfully!


# Movie Recommendation

A recommendation function is developed to generate similar movie suggestions based on the user's selected movie.

First, the selected movie is searched in the dataset to identify its index. The KNN model then uses the corresponding genre feature vector to find the nearest movies.

The `kneighbors()` function returns the distances and indices of the nearest movies. The first result is skipped because it represents the selected movie itself.

Finally, the system displays five recommended movies along with their genre information and similarity scores.

In [30]:
def recommend_movie(movie_name):

    # Find movie index
    movie_index = movies[
        movies['title'].str.lower() == movie_name.lower()
    ].index

    if len(movie_index) == 0:
        print("Movie not found!")
        return

    index = movie_index[0]

    # Find nearest neighbors
    distances, indices = knn_model.kneighbors(
        genre_features[index].reshape(1, -1)
    )

    print("Recommendations for:", movie_name)
    print("--------------------------------")

    for i, distance in zip(indices[0][1:], distances[0][1:]):

        similarity = (1 - distance) * 100

        print("Movie:", movies.iloc[i]['title'])
        print("Similarity:", round(similarity, 2), "%")
        print("--------------------------------")

In [31]:
recommend_movie("The Shawshank Redemption")

Recommendations for: The Shawshank Redemption
--------------------------------
Movie: The Godfather: Part II
Similarity: 100.0 %
--------------------------------
Movie: The Godfather
Similarity: 100.0 %
--------------------------------
Movie: Once Upon a Time in America
Similarity: 100.0 %
--------------------------------
Movie: Cherry
Similarity: 100.0 %
--------------------------------
Movie: Boys Don't Cry
Similarity: 100.0 %
--------------------------------


# Cosine Distance and Similarity Calculation

Cosine Distance is used to measure the difference between the genre feature vectors of two movies.

A lower cosine distance indicates that two movies have more similar genre features.

In this project, cosine distance is converted into a similarity percentage using the following formula:

**Similarity (%) = (1 − Cosine Distance) × 100**

For example, if the cosine distance between two movies is 0.20, their genre similarity is:

(1 − 0.20) × 100 = 80%

The similarity percentage helps display how closely the recommended movies match the genre features of the selected movie.

# Model Evaluation

The recommendation system is evaluated by calculating the average genre similarity of the five recommended movies.

The similarity scores of the recommended movies are calculated using cosine distance. The NumPy `mean()` function is then used to calculate the average similarity.

**Average Genre Similarity = (S1 + S2 + S3 + S4 + S5) / 5**

Where S1, S2, S3, S4, and S5 represent the similarity scores of the five recommended movies.

This evaluation provides an indication of how closely the recommended movies match the selected movie's genres. It is a similarity measure and not the actual prediction accuracy of the model.

In [32]:
def evaluate_recommendations(movie_name):

    movie_index = movies[
        movies['title'].str.lower() == movie_name.lower()
    ].index

    if len(movie_index) == 0:
        print("Movie not found!")
        return

    index = movie_index[0]

    distances, indices = knn_model.kneighbors(
        genre_features[index].reshape(1, -1)
    )

    scores = (1 - distances[0][1:]) * 100

    print("Movie:", movie_name)
    print("Average Genre Similarity:",
          round(np.mean(scores), 2), "%")

    print("Top 5 Recommendations:")
    for i, score in zip(indices[0][1:], scores):
        print(
            movies.iloc[i]['title'],
            "-",
            round(score, 2), "%"
        )

evaluate_recommendations("The Shawshank Redemption")

Movie: The Shawshank Redemption
Average Genre Similarity: 100.0 %
Top 5 Recommendations:
The Godfather: Part II - 100.0 %
The Godfather - 100.0 %
Once Upon a Time in America - 100.0 %
Cherry - 100.0 %
Boys Don't Cry - 100.0 %


# Model Saving Using Pickle

After training the KNN model, the model and its supporting data are saved using the Pickle library.

The `pickle.dump()` function is used to serialize and save Python objects into binary files.

The following files are generated:

- **movies_list.pkl:** Stores the processed movie dataset.
- **knn_model.pkl:** Stores the trained KNN model.
- **genre_features.pkl:** Stores the numerical genre feature matrix.

These saved files can be loaded later by the Streamlit application without fitting the KNN model again.

In [33]:
import pickle

# Save updated movie data
with open("movies_list.pkl", "wb") as f:
    pickle.dump(movies.reset_index(drop=True), f)

# Save KNN model
with open("knn_model.pkl", "wb") as f:
    pickle.dump(knn_model, f)

# Save genre features
with open("genre_features.pkl", "wb") as f:
    pickle.dump(genre_features, f)

print("All KNN files saved successfully!")

All KNN files saved successfully!


# Conclusion

The Netflix Movie Recommendation System successfully demonstrates the use of Machine Learning techniques for content-based movie recommendations. The dataset is preprocessed to remove missing values and duplicate movie IDs. Movie genres are converted into numerical features using One-Hot Encoding with MultiLabelBinarizer.

The K-Nearest Neighbors (KNN) algorithm with Cosine Distance is used to identify movies with similar genre characteristics. The system recommends the top five similar movies based on the selected movie and calculates their average genre similarity.

The trained model and supporting data are saved using Pickle and integrated with a Streamlit web application. This project provides a simple and effective way to discover movies based on genre similarity.